# Chapter 7: Vector Spaces

Companion notebook for *The Math That Powers AI* (2nd ed.), Chapter 7.

We import everything from the `mathpowersai.spaces` package module and walk through the chapter's worked examples:

1. Linear independence checks via matrix rank
2. Change of basis (the book's corrected example)
3. Gram-Schmidt orthonormalization
4. Orthogonal projection onto a vector and onto a subspace
5. The Word2Vec-style toy-embedding analogy demo (`king - man + woman ~= queen`)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

In [ ]:
from mathpowersai.spaces import (
    change_of_basis,
    embedding_analogy,
    gram_schmidt,
    is_linearly_independent,
    make_toy_embeddings,
    project_onto_subspace,
    project_onto_vector,
    projection_matrix,
)

rng = np.random.default_rng(42)

## 1. Linear independence

Vectors $v_1, \dots, v_k$ are **linearly independent** if the only solution to
$c_1 v_1 + \dots + c_k v_k = 0$ is $c_1 = \dots = c_k = 0$. Equivalently, the
matrix with these vectors as rows has rank $k$.

`is_linearly_independent` performs exactly this rank check. Below: a dependent
pair (one vector is a scalar multiple of the other), the standard basis of
$\mathbb{R}^3$, and a random set of vectors (random vectors are independent
with probability 1).

In [ ]:
v1, v2 = [1, 2, 3], [2, 4, 6]
print(f"v1={v1}, v2={v2} independent? "
      f"{is_linearly_independent([v1, v2])}")

e = np.eye(3)
print("standard basis e1,e2,e3 independent? "
      f"{is_linearly_independent(e)}")

R = rng.normal(size=(3, 5))  # 3 random vectors in R^5
print(f"3 random vectors in R^5 independent? "
      f"{is_linearly_independent(R)}")

# 4 vectors in R^3 can never be independent.
R4 = rng.normal(size=(4, 3))
print(f"4 random vectors in R^3 independent? "
      f"{is_linearly_independent(R4)}")

## 2. Change of basis

If $B = [\,b_1 \mid \dots \mid b_n\,]$ holds the basis vectors as columns, the
coordinates $c$ of a point $x$ with respect to that basis solve $Bc = x$, so
that $x = c_1 b_1 + \dots + c_n b_n$ (the unique representation theorem).

The book's example: the point $(1.5, 1.2)$ in the basis
$b_1 = (1.2, 0.4)$, $b_2 = (0.3, 1.0)$ has coordinates
$\approx (1.06, 0.78)_B$. We verify by reconstructing $x = B c$.

In [ ]:
B = np.column_stack([[1.2, 0.4], [0.3, 1.0]])
x = np.array([1.5, 1.2])

coords = change_of_basis(x, B)
print(f"point {x} in basis B -> coords {coords.round(2)}")

# Reconstruction check: B @ coords should give x back.
reconstructed = B @ coords
print(f"reconstruction B @ coords = {reconstructed.round(2)}")
assert np.allclose(reconstructed, x)
assert np.allclose(coords.round(2), [1.06, 0.78])
print("reconstruction matches the original point: True")

## 3. Gram-Schmidt orthonormalization

Given linearly independent vectors $\{v_1, \dots, v_k\}$, Gram-Schmidt produces
an orthonormal set $\{q_1, \dots, q_k\}$ with the same span:

$$u_j = v_j - \sum_{i=1}^{j-1} \langle v_j, q_i \rangle\, q_i, \qquad q_j = \frac{u_j}{\|u_j\|}.$$

The result $Q$ satisfies $Q^\top Q = I$. We run the chapter's worked example,
$v_1 = (1, 1, 0)$ and $v_2 = (1, 0, 1)$, then a larger random example.

In [ ]:
V = np.column_stack([[1.0, 1.0, 0.0], [1.0, 0.0, 1.0]])
Q = gram_schmidt(V)
print("Gram-Schmidt on v1=(1,1,0), v2=(1,0,1):")
print(f"q1 = {Q[:, 0].round(4)}")
print(f"q2 = {Q[:, 1].round(4)}")
print(f"<q1, q2> = {np.dot(Q[:, 0], Q[:, 1]):.6f}")  # ~0

# The key property: Q^T Q = I.
print(f"Q^T Q =\n{(Q.T @ Q).round(10)}")
assert np.allclose(Q.T @ Q, np.eye(2))

# A bigger random example: 4 vectors in R^6.
V_big = rng.normal(size=(6, 4))
Q_big = gram_schmidt(V_big)
print("random 6x4 case: Q^T Q == I? "
      f"{np.allclose(Q_big.T @ Q_big, np.eye(4))}")

## 4. Projection onto a vector and onto a subspace

**Onto a vector.** The orthogonal projection of $v$ onto $u$ is
$\mathrm{proj}_u(v) = \frac{\langle v, u \rangle}{\|u\|^2}\, u$, and the
residual $v - \mathrm{proj}_u(v)$ is perpendicular to $u$.

**Onto a subspace.** With the spanning vectors as the columns of $A$, the
projection matrix is $P = A (A^\top A)^{-1} A^\top$. $P$ is **idempotent**
($P^2 = P$): projecting twice changes nothing, because the projection already
lies in the subspace. The residual lies in the orthogonal complement
$W^\perp$.

In [ ]:
# Projection onto a vector (the chapter's printed listing).
v = np.array([2, 3])
u = np.array([4, 1])
proj = project_onto_vector(v, u)
residual = v - proj  # perpendicular component
print(f"v = {v}")
print(f"proj_u(v) = {proj.round(3)}")
print(f"residual = {residual.round(3)}")
print(f"check orthogonality <proj, residual>: "
      f"{np.dot(proj, residual):.6f}")  # ~0

# Projection onto the subspace W = span{(1,1,0), (1,0,1)}.
b = np.array([1.0, 2.0, 3.0])
p = project_onto_subspace(b, V)
r = b - p
print(f"\nproj_W({b}) = {p.round(4)}")
print(f"residual orthogonal to W? {np.allclose(V.T @ r, 0)}")

# Idempotence: P^2 = P, and projecting twice changes nothing.
P = projection_matrix(V)
print(f"P idempotent (P @ P == P)? {np.allclose(P @ P, P)}")
print(f"P symmetric (P.T == P)?    {np.allclose(P.T, P)}")
p_twice = project_onto_subspace(p, V)
print(f"proj_W(proj_W(b)) == proj_W(b)? {np.allclose(p_twice, p)}")

## 5. The toy-embedding analogy demo

The chapter's war story: in Word2Vec-style embedding spaces, semantic
attributes like "royalty" and "gender" become (approximately) separable
*directions* in the vector space. That is why vector arithmetic solves
analogies:

$$\text{king} - \text{man} + \text{woman} \approx \text{queen}.$$

`make_toy_embeddings` builds a tiny vocabulary with exactly this structure
(plus noise), and `embedding_analogy` finds the nearest word to the target
vector by cosine similarity, excluding the three query words.

In [ ]:
embeddings = make_toy_embeddings(rng)
print(f"vocabulary: {sorted(embeddings)}")
print(f"embedding dimension: {embeddings['king'].shape[0]}")

answer = embedding_analogy(embeddings, "king", "man", "woman")
print(f"king - man + woman ~= {answer}")
assert answer == "queen"

# The reverse analogy works too.
print(f"queen - woman + man ~= "
      f"{embedding_analogy(embeddings, 'queen', 'woman', 'man')}")